STRUCTURED OUTPUT — SHORT THEORY NOTES

1. What is Structured Output?
Structured output forces an LLM to return data in a strict format (usually JSON) matching a defined schema instead of free-form text.
- Why use it: Turns model output into reliable fields (strings, lists, ints) that code, APIs, or databases can use safely.
- Better than prompt-only JSON: Asking "please return JSON" in a prompt can still fail formatting; schema-constrained generation enforces the structure.

2. How the Pipeline Works
Input Text -> LLM Inference + JSON Schema (Constrained Decoding) -> Raw JSON -> Pydantic Validation -> Validated Python Object

- Pydantic: Defines the data contract (types, required fields, constraints like Field(ge=0)) and validates parsed output (raising ValidationError on bad types/values).
- JSON Schema: Converts the Pydantic model into a standard schema sent to the model/provider so generation stays aligned with the contract.
- Validated Python Object: Gives direct attribute access (e.g., obj.name, obj.skills) instead of manual string/JSON cleanup.

3. Key Catch (Format vs. Truth)
- Schema enforcement guarantees structural/type validity, NOT factual accuracy.
- Even with valid JSON and valid types, the LLM can still hallucinate or extract wrong facts.

In [1]:
!pip install -q transformers torch pydantic



In [4]:
from transformers import pipeline

generator =pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto"
)




config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [5]:
from pydantic import BaseModel

class student(BaseModel):
  name: str
  age: int
  branch:str
  skills:list[str]


In [7]:
prompt = """
Extract student information from this text.

Return only valid JSON with these fields:
name, age, branch, skills.

Text:
Rahul is 20 years old, studies CSE,
and knows Python, C++, and SQL.

JSON:
"""

result = generator(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)
raw_output=result[0]["generated_text"]
print(raw_output)

```json
{
  "name": "Rahul",
  "age": 20,
  "branch": "CSE",
  "skills": ["Python", "C++", "SQL"]
}
```

### Response:
```json
{
  "name": "Rahul",
  "age": 20,
  "branch": "CSE",
  "skills": ["Python", "C++", "SQL"]
}
```


In [9]:
import json

data = json.loads(raw_output)

student = Student.model_validate(data)

print(student)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [10]:
print(repr(raw_output))


'```json\n{\n  "name": "Rahul",\n  "age": 20,\n  "branch": "CSE",\n  "skills": ["Python", "C++", "SQL"]\n}\n```\n\n### Response:\n```json\n{\n  "name": "Rahul",\n  "age": 20,\n  "branch": "CSE",\n  "skills": ["Python", "C++", "SQL"]\n}\n```'


In [11]:
import re
import json

match = re.search(r"```(?:json)?\s*(.*?)```", raw_output, re.DOTALL)

if match:
    clean_output = match.group(1)
else:
    clean_output = raw_output.strip()

print(clean_output)

{
  "name": "Rahul",
  "age": 20,
  "branch": "CSE",
  "skills": ["Python", "C++", "SQL"]
}



In [13]:
data = json.loads(clean_output)

stud = student.model_validate(data)

print(stud)


name='Rahul' age=20 branch='CSE' skills=['Python', 'C++', 'SQL']


In [15]:
print("Name:", stud.name)
print("Age:", stud.age)
print("Branch:", stud.branch)
print("Skills:", stud.skills)

Name: Rahul
Age: 20
Branch: CSE
Skills: ['Python', 'C++', 'SQL']
